In [1]:
import pandas as pd
from pathlib import Path
from sklearn.ensemble import IsolationForest

processed_path = Path("../data/processed")

feature_data = pd.read_csv(
    processed_path / "daily_features.csv"
)

feature_data["date"] = pd.to_datetime(feature_data["date"])

evaluation_data = feature_data[
    feature_data["day_number"] > 7
].copy()

model_features = [
    "steps_deviation_7d_pct",
    "sleep_deviation_7d_pct"
]

# Hold out entire participants, not individual rows
participant_ids = sorted(evaluation_data["Id"].unique())

test_ids = participant_ids[-3:]

train_data = evaluation_data[
    ~evaluation_data["Id"].isin(test_ids)
].copy()

test_data = evaluation_data[
    evaluation_data["Id"].isin(test_ids)
].copy()

print("Train participants:", train_data["Id"].nunique())
print("Test participants:", test_data["Id"].nunique())
print("Train rows:", len(train_data))
print("Test rows:", len(test_data))
print("Test participant IDs:", test_ids)

Train participants: 12
Test participants: 3
Train rows: 65
Test rows: 9
Test participant IDs: [np.int64(7086361926), np.int64(8378563200), np.int64(8792009665)]


In [2]:
X_train = train_data[model_features]
X_test = test_data[model_features]

model = IsolationForest(
    n_estimators=100,
    contamination="auto",
    random_state=42
)

model.fit(X_train)

test_data["anomaly_label"] = model.predict(X_test)
test_data["anomaly_score"] = model.decision_function(X_test)

print("Test observations:", len(test_data))
print("\nPredictions:")
print(test_data["anomaly_label"].value_counts())

display(
    test_data[
        [
            "Id",
            "date",
            "steps_deviation_7d_pct",
            "sleep_deviation_7d_pct",
            "anomaly_label",
            "anomaly_score"
        ]
    ].sort_values("anomaly_score")
)

Test observations: 9

Predictions:
anomaly_label
-1    6
 1    3
Name: count, dtype: int64


,Id,date,steps_deviation_7d_pct,sleep_deviation_7d_pct,anomaly_label,anomaly_score
176,7086361926,2016-04-10,-93.127545,-87.755102,-1,-0.214813
178,7086361926,2016-04-12,-94.806405,-3.854875,-1,-0.100385
190,8378563200,2016-04-12,-68.580871,-49.413690,-1,-0.095372
188,8378563200,2016-04-10,-74.533663,19.498227,-1,-0.056152
198,8792009665,2016-04-09,-68.707900,29.037520,-1,-0.051541
187,8378563200,2016-04-09,-78.318284,9.571857,-1,-0.037800
186,8378563200,2016-04-08,-19.177533,-34.333242,1,0.032205
189,8378563200,2016-04-11,-14.463696,-26.506681,1,0.067199
177,7086361926,2016-04-11,-24.608324,16.326531,1,0.079745


In [3]:
test_data["rule_alert"] = (
    (test_data["steps_deviation_7d_pct"] <= -10)
    | (test_data["sleep_deviation_7d_pct"] <= -10)
)

ml_flag = test_data["anomaly_label"] == -1
rule_flag = test_data["rule_alert"]

print("ML anomalies:", ml_flag.sum())
print("Rule alerts:", rule_flag.sum())
print("Both flag:", (ml_flag & rule_flag).sum())
print("ML only:", (ml_flag & ~rule_flag).sum())
print("Rule only:", (~ml_flag & rule_flag).sum())

ML anomalies: 6
Rule alerts: 9
Both flag: 6
ML only: 0
Rule only: 3


## Day 11 — Participant-Level Model Evaluation

### Evaluation Setup
- Training set: 12 participants, 65 observations.
- Test set: 3 unseen participants, 9 observations.
- Isolation Forest was fitted exclusively on the training participants.
- Features: percentage deviation in daily steps and recorded sleep from personal baselines.

### Results
- Isolation Forest flagged 6 of 9 test observations as anomalies.
- The exploratory rule-based method flagged 9 of 9 observations.
- Both methods flagged 6 observations; 3 were flagged only by the rule.
- No observations were flagged exclusively by Isolation Forest.

### Limitations
- The test set is small and contains unequal participant observation counts.
- Ground-truth anomaly or clinical outcome labels are unavailable.
- Recorded sleep may reflect incomplete tracking rather than actual sleep duration.
- These results measure agreement between exploratory methods, not diagnostic accuracy or clinical validity.

### Conclusion
Participant-level holdout evaluation provides an initial check of anomaly detection on unseen participants. Larger datasets, stronger data-quality checks, repeated participant-level validation and independent outcome labels are needed before making performance or health-related claims.

In [1]:
import pandas as pd
from pathlib import Path

processed_path = Path("../data/processed")

feature_data = pd.read_csv(
    processed_path / "daily_features.csv"
)

feature_data["date"] = pd.to_datetime(feature_data["date"])

evaluation_data = feature_data[
    feature_data["day_number"] > 7
].copy()

candidate_features = [
    "steps_deviation_7d_pct",
    "sleep_deviation_7d_pct",
    "steps_rolling_3d",
    "sleep_rolling_3d",
    "Calories",
    "VeryActiveMinutes",
    "SedentaryMinutes"
]

X = evaluation_data[candidate_features]

print("Shape:", X.shape)
print("\nMissing values:")
print(X.isna().sum())

print("\nFeature statistics:")
display(X.describe().round(2))

print("\nFeature correlation:")
display(X.corr().round(2))

Shape: (74, 7)

Missing values:
steps_deviation_7d_pct    0
sleep_deviation_7d_pct    0
steps_rolling_3d          0
sleep_rolling_3d          0
Calories                  0
VeryActiveMinutes         0
SedentaryMinutes          0
dtype: int64

Feature statistics:


,steps_deviation_7d_pct,sleep_deviation_7d_pct,steps_rolling_3d,sleep_rolling_3d,Calories,VeryActiveMinutes,SedentaryMinutes
count,74.00,74.00,74.00,74.00,74.00,74.00,74.00
mean,-16.32,-4.11,8023.39,469.00,2209.72,17.70,688.50
std,37.88,31.78,3445.64,98.86,755.93,23.91,228.85
min,-99.22,-95.55,1112.67,180.33,600.00,0.00,61.00
25%,-37.77,-19.82,5501.75,401.25,1802.00,0.00,642.00
50%,-9.30,-2.00,7412.17,469.00,2188.00,7.50,731.50
75%,6.85,14.08,11099.50,532.33,2619.50,30.75,801.00
max,67.63,68.45,14480.67,685.00,3817.00,123.00,1368.00



Feature correlation:


,steps_deviation_7d_pct,sleep_deviation_7d_pct,steps_rolling_3d,sleep_rolling_3d,Calories,VeryActiveMinutes,SedentaryMinutes
steps_deviation_7d_pct,1.00,-0.08,0.43,0.05,0.45,0.14,0.27
sleep_deviation_7d_pct,-0.08,1.00,-0.10,0.35,-0.03,-0.05,-0.28
steps_rolling_3d,0.43,-0.10,1.00,-0.05,0.07,0.30,0.10
sleep_rolling_3d,0.05,0.35,-0.05,1.00,-0.18,-0.01,-0.27
Calories,0.45,-0.03,0.07,-0.18,1.00,0.28,0.47
VeryActiveMinutes,0.14,-0.05,0.30,-0.01,0.28,1.00,-0.09
SedentaryMinutes,0.27,-0.28,0.10,-0.27,0.47,-0.09,1.00


In [2]:
evaluation_data = evaluation_data.sort_values(
    ["Id", "date"]
).copy()

evaluation_data["steps_previous_3obs"] = (
    evaluation_data.groupby("Id")["TotalSteps"]
    .transform(
        lambda s: s.shift(1).rolling(3, min_periods=1).mean()
    )
)

evaluation_data["sleep_previous_3obs"] = (
    evaluation_data.groupby("Id")["recorded_sleep_minutes"]
    .transform(
        lambda s: s.shift(1).rolling(3, min_periods=1).mean()
    )
)

display(
    evaluation_data[
        [
            "Id",
            "date",
            "TotalSteps",
            "steps_previous_3obs",
            "recorded_sleep_minutes",
            "sleep_previous_3obs"
        ]
    ].head(10)
)

,Id,date,TotalSteps,steps_previous_3obs,recorded_sleep_minutes,sleep_previous_3obs
7,1503960366,2016-04-02,11248,NaN,591,NaN
8,1503960366,2016-04-03,10016,11248.000000,480,591.000000
9,1503960366,2016-04-04,14557,10632.000000,392,535.500000
10,1503960366,2016-04-05,14844,11940.333333,343,487.666667
11,1503960366,2016-04-09,12432,13139.000000,437,405.000000
12,1503960366,2016-04-10,10057,13944.333333,503,390.666667
13,1503960366,2016-04-11,10990,12444.333333,351,427.666667
25,1927972279,2016-04-10,2105,NaN,516,NaN
26,1927972279,2016-04-11,1209,2105.000000,559,516.000000
27,1927972279,2016-04-12,24,1657.000000,514,537.500000


In [3]:
print("Missing previous-step averages:",
      evaluation_data["steps_previous_3obs"].isna().sum())

print("Missing previous-sleep averages:",
      evaluation_data["sleep_previous_3obs"].isna().sum())

print("\nRows without previous observations:")
display(
    evaluation_data.loc[
        evaluation_data["steps_previous_3obs"].isna()
        | evaluation_data["sleep_previous_3obs"].isna(),
        ["Id", "date", "TotalSteps", "recorded_sleep_minutes"]
    ]
)

Missing previous-step averages: 15
Missing previous-sleep averages: 15

Rows without previous observations:


,Id,date,TotalSteps,recorded_sleep_minutes
7,1503960366,2016-04-02,11248,591
25,1927972279,2016-04-10,2105,516
35,2026352035,2016-04-08,3762,579
47,2347167796,2016-04-05,10114,510
61,3977333714,2016-04-08,10226,657
72,4020332650,2016-03-31,7428,281
87,4319703577,2016-04-05,8359,502
97,4445114986,2016-04-08,5543,251
111,4702921684,2016-04-05,6670,479
125,5553957443,2016-04-08,9938,458


In [4]:
from sklearn.ensemble import IsolationForest

improved_features = [
    "steps_deviation_7d_pct",
    "sleep_deviation_7d_pct",
    "steps_previous_3obs",
    "sleep_previous_3obs"
]

model_data = evaluation_data.dropna(
    subset=improved_features
).copy()

X_improved = model_data[improved_features]

print("Original observations:", len(evaluation_data))
print("Usable observations:", len(model_data))
print("Excluded observations:", len(evaluation_data) - len(model_data))
print("Missing values:\n", X_improved.isna().sum())

Original observations: 74
Usable observations: 59
Excluded observations: 15
Missing values:
 steps_deviation_7d_pct    0
sleep_deviation_7d_pct    0
steps_previous_3obs       0
sleep_previous_3obs       0
dtype: int64


In [5]:
improved_model = IsolationForest(
    n_estimators=100,
    contamination="auto",
    random_state=42
)

improved_model.fit(X_improved)

model_data["improved_label"] = improved_model.predict(X_improved)
model_data["improved_score"] = improved_model.decision_function(X_improved)

print("\nImproved model predictions:")
print(model_data["improved_label"].value_counts())

display(
    model_data[
        [
            "Id",
            "date",
            "steps_deviation_7d_pct",
            "sleep_deviation_7d_pct",
            "improved_label",
            "improved_score"
        ]
    ].sort_values("improved_score").head(10)
)


Improved model predictions:
improved_label
 1    45
-1    14
Name: count, dtype: int64


,Id,date,steps_deviation_7d_pct,sleep_deviation_7d_pct,improved_label,improved_score
177,7086361926,2016-04-11,-24.608324,16.326531,-1,-0.173198
51,2347167796,2016-04-09,62.925707,-95.549339,-1,-0.129031
27,1927972279,2016-04-12,-99.220237,54.885923,-1,-0.105033
52,2347167796,2016-04-10,2.105918,29.425229,-1,-0.075854
26,1927972279,2016-04-11,-60.719424,68.445975,-1,-0.069160
178,7086361926,2016-04-12,-94.806405,-3.854875,-1,-0.055266
115,4702921684,2016-04-09,67.628397,57.893258,-1,-0.029842
12,1503960366,2016-04-10,-22.756449,32.468021,-1,-0.020374
129,5553957443,2016-04-12,-62.873549,-34.804754,-1,-0.015776
78,4020332650,2016-04-10,-19.357157,-83.338694,-1,-0.011821


## Day 12 — Feature Engineering Experiment

### Objective
Investigate whether previous-observation activity and sleep trends can complement personal-baseline deviation features in anomaly detection.

### Features
- Steps deviation from the 7-observation personal baseline
- Recorded sleep deviation from the 7-observation personal baseline
- Previous 3 available observations' average steps
- Previous 3 available observations' average recorded sleep

### Results
- Original evaluation observations: 74
- Observations excluded due to unavailable previous-observation averages: 15
- Usable observations: 59
- Isolation Forest flagged 14 observations as anomalies and 45 as normal.

### Limitations
- The model was fitted and inspected on the same 59 observations; this is not independent evaluation.
- No ground-truth anomaly labels are available.
- Recorded sleep may be affected by incomplete wearable tracking.
- Results cannot establish clinical significance or prove that the expanded feature set is better.

### Conclusion
Previous-observation features provide an additional representation of recent activity and sleep patterns. A fair comparison requires the same participant-level train/test split and a consistent evaluation protocol for both feature sets.

In [6]:
import pandas as pd
from pathlib import Path

processed_path = Path("../data/processed")

feature_data = pd.read_csv(
    processed_path / "daily_features.csv"
)

feature_data["date"] = pd.to_datetime(feature_data["date"])

evaluation_data = feature_data[
    feature_data["day_number"] > 7
].copy()

evaluation_data = evaluation_data.sort_values(
    ["Id", "date"]
).copy()

evaluation_data["steps_previous_3obs"] = (
    evaluation_data.groupby("Id")["TotalSteps"]
    .transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
)

evaluation_data["sleep_previous_3obs"] = (
    evaluation_data.groupby("Id")["recorded_sleep_minutes"]
    .transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
)

# Same participant-level split used on Day 11
participant_ids = sorted(evaluation_data["Id"].unique())
test_ids = participant_ids[-3:]

train_data = evaluation_data[
    ~evaluation_data["Id"].isin(test_ids)
].copy()

test_data = evaluation_data[
    evaluation_data["Id"].isin(test_ids)
].copy()

print("Training participants:", train_data["Id"].nunique())
print("Test participants:", test_data["Id"].nunique())
print("Training rows:", len(train_data))
print("Test rows:", len(test_data))

Training participants: 12
Test participants: 3
Training rows: 65
Test rows: 9


In [7]:
from sklearn.ensemble import IsolationForest

basic_features = [
    "steps_deviation_7d_pct",
    "sleep_deviation_7d_pct"
]

expanded_features = basic_features + [
    "steps_previous_3obs",
    "sleep_previous_3obs"
]

# Use the same test observations for both models
common_test = test_data.dropna(
    subset=expanded_features
).copy()

basic_train = train_data.dropna(
    subset=basic_features
)

expanded_train = train_data.dropna(
    subset=expanded_features
)

basic_model = IsolationForest(
    n_estimators=100,
    contamination="auto",
    random_state=42
)

expanded_model = IsolationForest(
    n_estimators=100,
    contamination="auto",
    random_state=42
)

basic_model.fit(basic_train[basic_features])
expanded_model.fit(expanded_train[expanded_features])

common_test["basic_label"] = basic_model.predict(
    common_test[basic_features]
)

common_test["expanded_label"] = expanded_model.predict(
    common_test[expanded_features]
)

print("Common test observations:", len(common_test))
print("\nBasic model:")
print(common_test["basic_label"].value_counts())

print("\nExpanded model:")
print(common_test["expanded_label"].value_counts())

print(
    "\nBoth flagged:",
    (
        (common_test["basic_label"] == -1)
        & (common_test["expanded_label"] == -1)
    ).sum()
)

Common test observations: 6

Basic model:
basic_label
-1    4
 1    2
Name: count, dtype: int64

Expanded model:
expanded_label
-1    4
 1    2
Name: count, dtype: int64

Both flagged: 3


In [8]:
comparison = pd.crosstab(
    common_test["basic_label"],
    common_test["expanded_label"],
    rownames=["Basic model"],
    colnames=["Expanded model"]
)

display(comparison)

print(
    "Total matching predictions:",
    (
        common_test["basic_label"]
        == common_test["expanded_label"]
    ).sum(),
    "out of",
    len(common_test)
)

Expanded model,-1,1
Basic model,,
-1,3,1
1,1,1


Total matching predictions: 4 out of 6


## Day 13 — Model Comparison

Both feature sets flagged 4 of the 6 common test observations as anomalies.
Three observations were flagged by both models.

Equal anomaly counts do not establish equivalent predictions or superior performance. The small test set and absence of ground-truth labels limit conclusions. The comparison is exploratory, not clinical validation.